# PSF Training Notebook

This notebook reuses the existing training package from the repository to train the PSF basis-weight regressor against a generated dataset mounted at `/kaggle/input/psf-synthetic-dataset/generated`.

In [ ]:
# Repository setup
import os
import sys
from pathlib import Path

repo_candidates = [
    Path.cwd(),
    Path('/kaggle/working'),
    Path('/workspace'),
]

repo_root = None
for candidate in repo_candidates:
    if (candidate / 'train.py').exists() and (candidate / 'src' / 'training').exists():
        repo_root = candidate
        break

if repo_root is None:
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'Repository root: {repo_root}')

In [ ]:
# Install lightweight dependencies if needed
import subprocess
import sys

packages = ['pyyaml', 'pillow', 'torchvision']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
# Imports and config loading
import json
from pathlib import Path

import torch
import yaml
from matplotlib import pyplot as plt

from training.dataloader import build_dataloaders
from training.trainer import Trainer, TrainerConfig

config_path = repo_root / 'config' / 'train.yaml'
with config_path.open('r', encoding='utf-8') as handle:
    config_data = yaml.safe_load(handle)

dataset_root = Path('/kaggle/input/psf-synthetic-dataset/generated')
config_data['root_dir'] = str(dataset_root)
config_data['train_manifest'] = str(dataset_root / 'train.csv')
config_data['val_manifest'] = str(dataset_root / 'val.csv')
config_data['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config_data['checkpoint_dir'] = '/kaggle/working/checkpoints'
config_data['log_dir'] = '/kaggle/working/logs'

trainer_config = TrainerConfig(**config_data)
print('Using device:', trainer_config.device)

In [ ]:
# Create dataloaders using the existing repository helpers
train_loader, val_loader = build_dataloaders(
    root_dir=trainer_config.checkpoint_dir if False else dataset_root,
    train_manifest=trainer_config.train_manifest,
    val_manifest=trainer_config.val_manifest,
    batch_size=trainer_config.batch_size,
    num_workers=trainer_config.num_workers,
    target_dim=trainer_config.target_dim,
)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader) if val_loader is not None else 0}')

In [ ]:
# Train using the repository's trainer implementation
trainer = Trainer(trainer_config)
results = trainer.fit(train_loader, val_loader)

history = results['history']
print('Training complete.')
print(json.dumps(history[-1], indent=2))

In [ ]:
# Save training artifacts to /kaggle/working
output_dir = Path('/kaggle/working')
output_dir.mkdir(parents=True, exist_ok=True)

history_path = output_dir / 'training_history.json'
history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
print(f'Saved history to {history_path}')

In [ ]:
# Plot training and validation loss
epochs = [entry['epoch'] for entry in history]
train_loss = [entry['train_loss'] for entry in history]
val_loss = [entry['val_loss'] for entry in history if not isinstance(entry['val_loss'], float) or entry['val_loss'] == entry['val_loss']]

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_loss, label='train_loss', marker='o')
if len(val_loss) == len(epochs):
    plt.plot(epochs, val_loss, label='val_loss', marker='x')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()